In [1]:
import os
import sqlite3
import pandas as pd

# Path resolution handling (works from root or notebooks/ folder)
users_path = '../data/processed/Cleaned_Social_Engine_Users.csv' if os.path.exists('../data/processed/Cleaned_Social_Engine_Users.csv') else 'data/processed/Cleaned_Social_Engine_Users.csv'
posts_path = '../data/processed/Cleaned_Social_Engine_Posts.csv' if os.path.exists('../data/processed/Cleaned_Social_Engine_Posts.csv') else 'data/processed/Cleaned_Social_Engine_Posts.csv'

# Connect to database
conn = sqlite3.connect("data_vortex.db")

# Load cleaned CSVs into SQL tables
df_users = pd.read_csv(users_path)
df_posts = pd.read_csv(posts_path)

df_users.to_sql("users", conn, if_exists="replace", index=False)
df_posts.to_sql("posts", conn, if_exists="replace", index=False)

print("Tables 'users' and 'posts' loaded successfully!")

Tables 'users' and 'posts' loaded successfully!


## Query E3 (Easy Level)
### E3: Average Engagement by Platform

In [3]:
query_e3 = """
WITH PlatformMetrics AS (
    SELECT 
        platform,
        COUNT(post_id) AS total_posts,
        AVG(COALESCE(likes, 0)) AS avg_likes,
        AVG(COALESCE(shares, 0)) AS avg_shares,
        AVG(COALESCE(comments, 0)) AS avg_comments,
        AVG(COALESCE(likes, 0) + COALESCE(shares, 0) + COALESCE(comments, 0)) AS avg_total_engagement
    FROM posts
    WHERE platform IS NOT NULL AND TRIM(platform) != ''
    GROUP BY platform
)
SELECT 
    platform,
    ROUND(avg_likes, 2) AS avg_likes,
    ROUND(avg_shares, 2) AS avg_shares,
    ROUND(avg_comments, 2) AS avg_comments,
    ROUND(avg_total_engagement, 2) AS avg_total_engagement
FROM PlatformMetrics
ORDER BY avg_total_engagement DESC;
"""

df_e3 = pd.read_sql_query(query_e3, conn)
df_e3

,platform,avg_likes,avg_shares,avg_comments,avg_total_engagement
0,YouTube,2528.34,1011.83,504.38,4044.55
1,Instagram,2500.19,1040.84,499.80,4040.83
2,Facebook,2535.65,984.17,506.94,4026.76
3,Reddit,2488.95,1002.23,511.18,4002.36
4,Unknown,2467.08,998.61,496.54,3962.22
5,Twitter,2437.74,1005.39,506.13,3949.26


## Query M1 (Medium Level)
### M1: Location Engagement Analysis

In [4]:
query_m1 = """
WITH LocationEngagement AS (
    SELECT 
        COALESCE(u.location, 'Unknown') AS location,
        COUNT(p.post_id) AS post_count,
        SUM(COALESCE(p.likes, 0) + COALESCE(p.shares, 0) + COALESCE(p.comments, 0)) AS total_engagement,
        ROUND(AVG(COALESCE(p.likes, 0) + COALESCE(p.shares, 0) + COALESCE(p.comments, 0)), 2) AS avg_engagement_per_post
    FROM users u
    JOIN posts p ON u.user_id = p.user_id
    GROUP BY COALESCE(u.location, 'Unknown')
)
SELECT 
    location,
    post_count,
    total_engagement,
    avg_engagement_per_post,
    DENSE_RANK() OVER (ORDER BY total_engagement DESC) AS engagement_rank
FROM LocationEngagement
ORDER BY engagement_rank ASC;
"""

df_m1 = pd.read_sql_query(query_m1, conn)
df_m1

,location,post_count,total_engagement,avg_engagement_per_post,engagement_rank
0,"Los Angeles, USA",459,1838884,4006.28,1
1,"Munich, Germany",452,1812310,4009.54,2
2,"Shanghai, China",451,1795736,3981.68,3
3,"Barcelona, Spain",439,1788641,4074.35,4
4,"Melbourne, Australia",421,1717585,4079.77,5
5,"Dubai, UAE",421,1706751,4054.04,6
6,"Houston, USA",422,1669232,3955.53,7
7,"Osaka, Japan",398,1640653,4122.24,8
8,"Rio de Janeiro, Brazil",414,1630322,3937.98,9
9,"Mumbai, India",413,1615304,3911.15,10


## Query H4 (Hard Level)
### H4: Follower-to-Engagement Anomaly Detection

In [5]:
query_h4 = """
WITH UserAggregates AS (
    SELECT 
        u.user_id,
        u.location,
        u.follower_count,
        COUNT(p.post_id) AS total_posts,
        SUM(COALESCE(p.likes, 0) + COALESCE(p.shares, 0) + COALESCE(p.comments, 0)) AS total_user_engagement
    FROM users u
    JOIN posts p ON u.user_id = p.user_id
    GROUP BY u.user_id, u.location, u.follower_count
),
RankedUsers AS (
    SELECT 
        user_id,
        location,
        follower_count,
        total_posts,
        total_user_engagement,
        NTILE(10) OVER (ORDER BY total_user_engagement DESC) AS engagement_decile
    FROM UserAggregates
)
SELECT 
    user_id,
    location,
    follower_count,
    total_posts,
    total_user_engagement,
    engagement_decile
FROM RankedUsers
WHERE follower_count < 5000 
  AND engagement_decile = 1
ORDER BY total_user_engagement DESC;
"""

df_h4 = pd.read_sql_query(query_h4, conn)
df_h4

,user_id,location,follower_count,total_posts,total_user_engagement,engagement_decile
0,user_uerv85na,"Rome, Italy",1824,16,72361,1
1,user_hdas0iau,"Rio de Janeiro, Brazil",1620,16,64007,1
2,user_n0ok02rt,"Dubai, UAE",2531,18,62920,1
3,user_fgjkkrie,"Lyon, France",2211,14,62690,1
4,user_6wra58f7,"Johannesburg, South Africa",4953,14,57476,1
5,user_5oe5t3js,"London, UK",3151,13,55241,1
6,user_rr1uzkql,"Milan, Italy",1069,14,54632,1
7,user_r7eg1rac,"Houston, USA",2052,12,54552,1
8,user_ogtvuuki,"Cairo, Egypt",4459,11,52208,1
9,user_67hyf45u,"Vancouver, Canada",898,11,52098,1
